# Day 78: Capstone Part 3 - Deployment & Wrap-up

## Reload and Finalize the Model
Part 2 compared several models and tuned a Random Forest using only the training split, holding out test data to get an honest performance estimate - but a model meant for deployment should ultimately be trained on all available data, since more data generally means a better final model and we've already measured its expected performance. Rather than saving the scaler and model as two separate objects (which risks someone loading the model but forgetting to apply the matching scaler, or applying the wrong version of it), we bundle them into a single scikit-learn `Pipeline`: preprocessing (`StandardScaler`) and modeling (`RandomForestClassifier`) become one object that can be fit, saved, loaded, and used for prediction as a unit. Calling `pipeline.fit(X, y)` on the full dataset fits both steps together - the scaler learns its mean/variance from all the data, then the Random Forest trains on the scaled output - producing one final, ready-to-deploy artifact.</cell id="cell-1">


In [ ]:
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

data = load_wine(as_frame=True)
df = data.frame
X = df.drop(columns=['target'])
y = df['target']

pipeline = Pipeline([
    ('scaler', StandardScaler()),  # step 1: scale features
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))  # step 2: classify using the scaled features
])
pipeline.fit(X, y)  # fits both steps together, trained on the FULL dataset for the final deployable model
print('Pipeline trained and ready to save')

## Saving the Full Pipeline
This is the same `joblib.dump` pattern from the model-deployment basics notebook, but applied to the whole `Pipeline` object instead of just the raw model - this is an important detail for real deployments. If you saved only the trained Random Forest and separately saved the scaler, anyone loading them later would have to remember to apply them in the correct order, and could easily use a mismatched or stale scaler. By saving the fitted Pipeline as a single object, `wine_pipeline.pkl` now contains everything needed to go from raw, unscaled input features straight to a prediction in one `.predict()` call, with zero risk of preprocessing/model mismatches.</cell id="cell-3">


In [ ]:
import joblib

joblib.dump(pipeline, 'wine_pipeline.pkl')  # saves the scaler + model together as one deployable artifact
print('Saved to wine_pipeline.pkl')

## Deployment Script
This closes the loop on the whole capstone: the same Streamlit pattern from the deployment-basics notebook, now wired up to the saved Wine pipeline instead of the earlier Iris model. Because we saved the full Pipeline (scaler + model together), the app can load one object and immediately call `.predict()` on raw feature values without needing to separately re-apply any scaling logic. This is a skeleton rather than a fully working app - the Wine dataset has 13 features, and a real version of this app would need a slider or input for each one (only `alcohol` is shown here as an example) plus a "Predict" button that collects all the inputs into a single feature array and calls `pipeline.predict()` on it, following the same button-click pattern shown in the earlier Streamlit example.</cell id="cell-5">


In [ ]:
# pip install streamlit
# app.py
import streamlit as st
import joblib
import numpy as np

pipeline = joblib.load('wine_pipeline.pkl')  # loads scaler + model together, ready for raw-feature predictions
st.title('Wine Classifier')
alcohol = st.slider('Alcohol', 10.0, 15.0, 13.0)  # example input widget for 1 of the 13 wine features
# ... collect the other 12 features similarly in a real app
st.write('Fill in all features, then click Predict to classify the wine cultivar')
# Run with: streamlit run app.py

## Capstone Wrap-up Checklist
Stepping back from the code, it's worth summarizing the shape of a complete data science project, since that overall process matters as much as any individual technique used along the way. Each item below corresponds to a stage we actually walked through across the three capstone notebooks: defining the problem and target classes (Part 1), exploring and understanding the data (Part 1), comparing multiple candidate models and tuning the best one (Part 2), honestly reporting final metrics on held-out data rather than cherry-picking favorable numbers (Part 2), and packaging the model into something deployable (Part 3). The last item - summarizing findings for a non-technical audience - is easy to skip but is often what actually determines whether a project's results get used, since stakeholders typically care about the practical takeaway (how well it works, what it enables) more than the modeling details.</cell id="cell-7">


In [ ]:
checklist = [
    'Problem clearly defined',
    'Data explored and cleaned',
    'Multiple models compared',
    'Best model tuned',
    'Final metrics reported honestly',
    'Model saved and deployable',
    'Findings summarized for a non-technical audience'
]
for item in checklist:
    print('[ ]', item)